# 📘 Notebook 8: Bayes' Theorem — Derivation and ML Applications

**Week 2 — Calculus, Optimization & Probability Theory**

**Difficulty:** ⭐⭐⭐ (Intermediate)

---

## 🏠 Why Does This Matter?

You built a house price model. A client says: "I think this house is worth $500k."  
You run your model and it says: "The features suggest $350k."

**Whose estimate should you trust? How do you combine both?**

Bayes' theorem tells you **exactly how to update your belief when you receive new evidence**.

This is the most important equation in ML:
- **Bayesian neural networks** → uncertainty-aware predictions
- **MAP estimation** → MLE with regularization
- **Naive Bayes classifier** → spam filter, medical diagnosis
- **Why L2 regularization works** → it's a Gaussian prior on weights!

---

## 📚 Prerequisites
- Notebooks 6–7 (probability spaces, conditional probability)

---

## Part 1: Derivation of Bayes' Theorem

### Plain English First

You have:
- A **prior belief**: what you think BEFORE seeing the data (P(hypothesis))
- **Evidence**: the data you observe
- A **likelihood**: how probable is this evidence IF your hypothesis is true? (P(data | hypothesis))
- A **posterior**: your updated belief AFTER seeing the data (P(hypothesis | data))

**Bayes says:** posterior ∝ likelihood × prior

### Derivation (2 lines!)

Start from conditional probability:
$$P(A|B) = \frac{P(A \cap B)}{P(B)} \quad \Rightarrow \quad P(A \cap B) = P(A|B) \cdot P(B)$$

Also: $P(A \cap B) = P(B|A) \cdot P(A)$

Equate and rearrange:

$$\boxed{P(A|B) = \frac{P(B|A) \cdot P(A)}{P(B)}}$$

In ML language:
$$\underbrace{P(\theta | \text{data})}_{\text{posterior}} = \frac{\overbrace{P(\text{data}|\theta)}^{\text{likelihood}} \cdot \overbrace{P(\theta)}^{\text{prior}}}{\underbrace{P(\text{data})}_{\text{normalizing constant}}}$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm, beta

# ─────────────────────────────────────────────────────────────────
# HOUSE PRICE EXAMPLE: Bayes' Theorem
#
# Question: A house has 4 bedrooms. Is it expensive (>$400k)?
#
# Prior:      P(expensive) = 0.25  (25% of all houses are expensive)
# Likelihood: P(4 bedrooms | expensive) = 0.35  (35% of expensive houses have 4 beds)
#             P(4 bedrooms | not expensive) = 0.18 (18% of cheap houses have 4 beds)
# Evidence:   P(4 bedrooms) = ? (use law of total probability)
#
# Posterior:  P(expensive | 4 bedrooms) = ?
# ─────────────────────────────────────────────────────────────────

# Prior
P_expensive     = 0.25     # 25% of all houses are expensive
P_not_expensive = 0.75     # 75% are not expensive

# Likelihood: how common is "4 bedrooms" in each class?
P_4beds_given_expensive     = 0.35
P_4beds_given_not_expensive = 0.18

# Evidence: total P(4 bedrooms) via Law of Total Probability
P_4beds = (P_4beds_given_expensive     * P_expensive +
           P_4beds_given_not_expensive * P_not_expensive)

# Posterior: P(expensive | 4 bedrooms)
P_expensive_given_4beds = (P_4beds_given_expensive * P_expensive) / P_4beds

print("BAYES' THEOREM: House Price Classification")
print("=" * 55)
print(f"  Prior P(expensive)                     = {P_expensive:.3f}")
print(f"  Likelihood P(4 beds | expensive)        = {P_4beds_given_expensive:.3f}")
print(f"  Likelihood P(4 beds | not expensive)    = {P_4beds_given_not_expensive:.3f}")
print()
print(f"  Evidence P(4 beds) = {P_4beds_given_expensive}×{P_expensive} + {P_4beds_given_not_expensive}×{P_not_expensive}")
print(f"                     = {P_4beds_given_expensive*P_expensive:.4f} + {P_4beds_given_not_expensive*P_not_expensive:.4f}")
print(f"                     = {P_4beds:.4f}")
print()
print(f"  Posterior P(expensive | 4 beds) = {P_4beds_given_expensive:.3f} × {P_expensive:.3f} / {P_4beds:.4f}")
print(f"                                  = {P_expensive_given_4beds:.4f}")
print()
print(f"  Before seeing bedrooms:  P(expensive) = {P_expensive:.3f}  (25%)")
print(f"  After learning it has 4 beds: P(expensive|4 beds) = {P_expensive_given_4beds:.3f}  ({P_expensive_given_4beds*100:.1f}%)")
print(f"  The evidence UPDATED our belief from {P_expensive*100:.0f}% to {P_expensive_given_4beds*100:.1f}%!")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Sequential Bayesian updating: adding features one at a time
# Each new piece of information updates the posterior
# The posterior from step t becomes the PRIOR for step t+1
# ─────────────────────────────────────────────────────────────────

# We'll add three features sequentially:
# Feature 1: 4+ bedrooms (P(feature|expensive)=0.35, P(feature|cheap)=0.18)
# Feature 2: >2000 sqft  (P(feature|expensive)=0.70, P(feature|cheap)=0.30)
# Feature 3: New build    (P(feature|expensive)=0.45, P(feature|cheap)=0.25)

features = [
    ("4+ bedrooms",  0.35, 0.18),
    (">2000 sqft",   0.70, 0.30),
    ("new build",    0.45, 0.25),
]

prior = P_expensive   # start with the baseline probability
belief_history = [('(no info)', prior)]

print("Sequential Bayesian Updating")
print(f"Starting prior: P(expensive) = {prior:.4f}")
print()

for feat_name, P_feat_given_exp, P_feat_given_cheap in features:
    # Apply Bayes' theorem to update the belief
    P_exp        = prior
    P_nexp       = 1 - prior
    # P(feature) via total probability
    P_feat       = P_feat_given_exp * P_exp + P_feat_given_cheap * P_nexp
    # Posterior via Bayes
    posterior    = P_feat_given_exp * P_exp / P_feat

    print(f"After observing '{feat_name}':")
    print(f"  P(expensive | this feature) = {prior:.4f} → {posterior:.4f}")

    belief_history.append((feat_name, posterior))
    prior = posterior   # posterior becomes prior for next step!

# ─── Plot the belief evolution ─────────────────────────────────────
names   = [h[0] for h in belief_history]
beliefs = [h[1] for h in belief_history]

plt.figure(figsize=(10, 5))
plt.plot(range(len(beliefs)), beliefs, 'steelblue', marker='o', markersize=10,
         linewidth=2.5, zorder=5)
plt.axhline(P_expensive, color='gray', linestyle=':', linewidth=1.5, label='Original prior')
for i, (name, belief) in enumerate(belief_history):
    plt.annotate(f'{belief:.3f}', (i, belief),
                 textcoords='offset points', xytext=(0, 12), ha='center', fontsize=11)
plt.xticks(range(len(names)), names, fontsize=10)
plt.ylabel('P(house is expensive)', fontsize=11)
plt.ylim(0, 1)
plt.title('Bayesian Updating: Each Feature Shifts Our Belief About House Price',
          fontsize=12)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=10)
plt.tight_layout()
plt.show()

print(f"\nFinal posterior: P(expensive | all 3 features) = {beliefs[-1]:.4f}")
print(f"Compared to prior: {P_expensive:.4f}  → belief went from {P_expensive*100:.1f}% to {beliefs[-1]*100:.1f}%")

## Part 2: Why L2 Regularization IS Bayes' Theorem

### Plain English First

This is one of the most beautiful connections in ML:

When you add L2 regularization to your house price model:
$$L_{\text{reg}}(w) = \underbrace{\frac{1}{N}\|Xw-y\|^2}_{\text{fit the data}} + \underbrace{\lambda \|w\|^2}_{\text{keep weights small}}$$

You are **actually doing Bayesian estimation** with a **Gaussian prior** on the weights!  
The prior says: "I believe the optimal weights are probably close to 0."

- **MLE** = maximize P(data | weights) = minimize MSE  
- **MAP** = maximize P(weights | data) ∝ P(data | weights) × P(weights) = minimize MSE + L2
- **λ** = how strongly you believe the prior (small λ → trust data; large λ → trust prior)

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Demonstrate: MLE vs MAP with different regularization strengths
# House price prediction with intentionally small dataset (to see overfitting)
# ─────────────────────────────────────────────────────────────────

np.random.seed(42)
N_train = 20    # intentionally small: 20 houses for training
N_test  = 500
N_feats = 5     # 5 features (sqft, beds, age, location, garage)

# Generate data with true weights near 0
w_true = np.array([2.0, -1.0, 0.5, 1.5, -0.3])

X_train = np.random.randn(N_train, N_feats)
y_train = X_train @ w_true + 0.5 * np.random.randn(N_train)
X_test  = np.random.randn(N_test, N_feats)
y_test  = X_test  @ w_true + 0.5 * np.random.randn(N_test)

def fit_mse_with_regularization(X, y, lambda_l2):
    """
    Closed-form solution for linear regression with L2 regularization.

    No regularization (λ=0): MLE solution  → w = (X.T X)⁻¹ X.T y
    With regularization (λ>0): MAP solution → w = (X.T X + λI)⁻¹ X.T y

    Adding λI to X.T X is the "ridge regression" trick.
    Bayesian interpretation: λ = σ²/σ²_prior (ratio of noise to prior variance)
    """
    n_feats = X.shape[1]
    A = X.T @ X + lambda_l2 * np.eye(n_feats)  # regularized normal equations
    b = X.T @ y
    return np.linalg.solve(A, b)   # solve the linear system: A @ w = b

lambda_values = [0.0, 0.01, 0.1, 1.0, 10.0]

print(f"{'λ (prior strength)':20} | {'Train MSE':12} | {'Test MSE':12} | {'‖w‖ (weight norm)':18}")
print("-" * 70)
for lam in lambda_values:
    w_fitted  = fit_mse_with_regularization(X_train, y_train, lam)
    train_mse = np.mean((X_train @ w_fitted - y_train)**2)
    test_mse  = np.mean((X_test  @ w_fitted - y_test)**2)
    w_norm    = np.linalg.norm(w_fitted)
    label     = "(pure MLE)" if lam == 0 else "(MAP)"
    print(f"  λ={lam:7}  {label:12} | {train_mse:12.4f} | {test_mse:12.4f} | {w_norm:18.4f}")

print()
print("Observations:")
print("  λ=0 (MLE):  Lowest train MSE, but HIGHEST test MSE → overfitting!")
print("  λ=0.1:      Good balance — lower test MSE → better generalization")
print("  λ=10:       Too strong prior → underfitting (weights forced too small)")
print()
print("Bayesian interpretation: λ controls how much we trust the data vs the prior.")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Naive Bayes Classifier for House Price Category
#
# "Naive" because it assumes features are independent given the class.
# Still works well in practice!
# ─────────────────────────────────────────────────────────────────

np.random.seed(42)
N_HOUSES = 100_000

# Recreate dataset
log_prices = np.random.normal(loc=np.log(300_000), scale=0.5, size=N_HOUSES)
prices     = np.exp(log_prices)
bedrooms   = np.random.choice([1,2,3,4,5,6], size=N_HOUSES, p=[0.05,0.25,0.40,0.20,0.08,0.02])
# Add sqft feature: correlated with price
sqft = 1000 + 300 * (np.log(prices/300_000) + np.random.randn(N_HOUSES)*0.3)
sqft = np.clip(sqft, 500, 5000).astype(int)

# Binary target: expensive (>$400k) or not
y = (prices > 400_000).astype(int)  # 0 = not expensive, 1 = expensive

# Binary features for Naive Bayes
f1_large_sqft = (sqft > 2000).astype(int)   # feature 1: large square footage
f2_many_beds  = (bedrooms >= 4).astype(int) # feature 2: 4+ bedrooms

# ─── Train Naive Bayes (estimate all conditional probabilities) ────
P_class = {0: np.mean(y==0), 1: np.mean(y==1)}   # P(class=0) and P(class=1)

# P(feature=1 | class) for each feature and class
features_list = [f1_large_sqft, f2_many_beds]
feat_names    = ['large sqft', 'many beds']
P_feat_given_class = {}
for c in [0, 1]:
    mask = (y == c)
    for fi, (feat, name) in enumerate(zip(features_list, feat_names)):
        P_feat_given_class[(c, fi)] = np.mean(feat[mask])  # P(feat=1 | class=c)

def naive_bayes_predict(feat_values):
    """
    Predict P(class | features) using Naive Bayes.

    Naive Bayes assumption: features are independent given class.
    P(class | f1, f2) ∝ P(f1|class) × P(f2|class) × P(class)

    feat_values: list of 0 or 1 for each feature
    Returns: dict {0: P(not expensive|features), 1: P(expensive|features)}
    """
    log_posteriors = {}    # use log-probabilities for numerical stability
    for c in [0, 1]:
        log_p = np.log(P_class[c])    # start with log-prior
        for fi, fval in enumerate(feat_values):
            p_feat = P_feat_given_class[(c, fi)]   # P(feat=1 | class)
            if fval == 1:
                log_p += np.log(p_feat + 1e-10)       # P(feat=1 | class)
            else:
                log_p += np.log(1 - p_feat + 1e-10)   # P(feat=0 | class)
        log_posteriors[c] = log_p

    # Convert from log-posteriors to proper probabilities (normalize)
    max_log = max(log_posteriors.values())
    unnorm  = {c: np.exp(lp - max_log) for c, lp in log_posteriors.items()}
    total   = sum(unnorm.values())
    return {c: v/total for c, v in unnorm.items()}

# Test on all 4 possible feature combinations
print("Naive Bayes House Price Classifier")
print(f"P(expensive) = {P_class[1]:.3f}")
print()
print(f"{'Features':35} | {'P(not exp.)':12} | {'P(expensive)':14} | Prediction")
print("-" * 80)
for f1, f2 in [(0,0),(0,1),(1,0),(1,1)]:
    desc  = f"large_sqft={'yes' if f1 else 'no':3}, many_beds={'yes' if f2 else 'no'}"
    probs = naive_bayes_predict([f1, f2])
    pred  = 'EXPENSIVE' if probs[1] > 0.5 else 'not expensive'
    print(f"  {desc}  | {probs[0]:.4f}       | {probs[1]:.4f}         | {pred}")

---

## ✅ Self-Check Questions

**1.** Write Bayes' theorem in words, replacing the math with the house price context:  
   P(expensive | large sqft) = ...

**2.** Prior: P(expensive) = 0.20. You observe the house is new build.  
   P(new build | expensive) = 0.40, P(new build | not expensive) = 0.15.  
   What is the posterior P(expensive | new build)?

**3.** In MAP estimation, what does it mean to have **λ → ∞** (very large regularization)?  
   What would the model's weights look like?

**4.** A Naive Bayes classifier computes P(expensive | big sqft, many beds).  
   It uses: P(big sqft|expensive) × P(many beds|expensive) × P(expensive).  
   What assumption makes this valid?

**5.** Your Bayesian model has a strong prior that prices are around $300k.  
   You observe 3 houses that all sold for $600k.  
   Will the posterior be closer to $300k or $600k? What if you observe 300 such houses?

<details>
<summary>Click to see answers</summary>

1. P(expensive | large sqft) = P(large sqft | expensive) × P(expensive) / P(large sqft). In words: "The probability of an expensive house given large sqft equals how often expensive houses have large sqft, times the base rate of expensive houses, divided by how common large sqft is overall."

2. P(new build) = 0.40×0.20 + 0.15×0.80 = 0.08 + 0.12 = 0.20. Posterior = 0.40×0.20/0.20 = **0.40**.

3. With λ→∞, the prior completely dominates. All weights → 0. The model predicts the mean price for every house, ignoring features entirely.

4. The **conditional independence** assumption: given that the house is expensive, the sqft feature and the bedroom feature are independent (knowing one doesn't tell you about the other).

5. With 3 houses: posterior will be between $300k and $600k, probably closer to $300k (prior dominates). With 300 houses: posterior will be much closer to $600k (data overwhelms the prior with enough evidence).

</details>

---

## 📌 Summary

| Term | Formula | House price meaning |
|------|---------|--------------------|
| **Prior** | P(θ) | What we believe about prices BEFORE data |
| **Likelihood** | P(data\|θ) | How probable is this data if θ is true? |
| **Posterior** | P(θ\|data) | Updated belief after seeing the houses |
| **Evidence** | P(data) | Normalizing constant (makes posterior sum to 1) |
| **MLE** | argmax P(data\|θ) | Best-fit model ignoring prior beliefs |
| **MAP** | argmax P(θ\|data) | Best model incorporating prior (= MLE + regularization!) |

**➡️ Next Notebook:** Common Probability Distributions — the mathematical shapes that describe data and model outputs.